# Data

In [1]:
import numpy as np
import pandas  as pd
import os
from IPython.display import clear_output
import pygad
from classes import MACDBacktester, backtest

os.chdir(r'c:\\Users\\arvin\\Documents\\Coding Project\\V4\\Algotrading_RL\\src')
from utils.utils import CreateTimeFrames

os.chdir(r'c:\\Users\\arvin\\Documents\\Coding Project\\V4\\Algotrading_RL\\src\\data')
#os.getcwd()

df = pd.read_csv('ETHUSD_2023_2024.csv',
                  index_col = 'Gmt time')
df.index = pd.to_datetime(df.index, format='%d.%m.%Y %H:%M:%S.%f', errors='coerce')
df = df.rename(columns={'Date':'time', 'Open': 'open', 'High': 'high', 'Low': 'low', 'Close':'close', 'Volume':'volume'})

os.chdir(r'c:\\Users\\arvin\\Documents\\Coding Project\\V4\\Algotrading_RL\\src\\genetic')

price_column = 'close'
date_split = "2024-10-1"
tf = '4h'

timeframes = ['1min','5min','15min', '30min','1h', '4h','1d','1w','1m']
df = CreateTimeFrames(df,timeframes)

working_dataset = df[tf]

working_dataset = working_dataset.iloc[-4050:,:]

train_end_index = int(len(working_dataset)*0.8)
TRAIN_END_DATE = working_dataset.index[train_end_index]

df_train = working_dataset.loc[:TRAIN_END_DATE]
df_train = df_train.drop(index=df_train.index[-1])


df_test = working_dataset.loc[TRAIN_END_DATE:,:]

assert len(df_train)+len(df_test)== len(working_dataset), "Some data are missed."


signal_price = 'smoothed_data'
real_price = 'close'

SEQ_LENGTH = 28

clear_output()


# Wavelet

In [2]:
os.getcwd()
os.chdir('..')

In [3]:
from denoise.dwt import *

In [4]:
df_train.loc[:,'smoothed_data'] = wavelet_denoising(df_train['close'], level=2)
df_test.loc[:,'smoothed_data'] = wavelet_denoising(df_test['close'], level=2)

C:\Users\arvin\AppData\Local\Temp\ipykernel_4216\3697563103.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test.loc[:,'smoothed_data'] = wavelet_denoising(df_test['close'], level=2)


# Test

## Backtester class

In [5]:
class MACDBacktester:
    def __init__(self, data, fast_ema, slow_ema, signal_line, signal_price = 'close', real_price = 'close',
                  sell_fee = 0.115, buy_fee = 0.115, initial_capital = 100):
        """"
        It works as a trade andlyzer with an specific amount of money and trading cost.

        :param data: Dataframe contains different features
        :param fast_ema: Integer
        :param slow_ema: Ineger
        :param signal_line: Integer
        :param signal_price: for producing buy and sell signals(defualt:'close')
        :param real_price: for trading(defualt:'close')
        :param sell_fee: Float contains sell trading cost(defulat=0.115). Nobitex maker trading cost is 0.1 and taker is 0.13.
        :param buy_fee: Float contains buy trading cost(defulat=0.115). Nobitex
        :param initial_capital: Integer contains initial money to start.

        """
        self.data = data.copy()
        self.fast_ema = fast_ema
        self.slow_ema = slow_ema
        self.signal_line = signal_line
        self.signal_price = signal_price
        self.real_price = real_price
        self.sell_fee_percent = sell_fee / 100
        self.buy_fee_percent = buy_fee / 100
        self.initial_capital = initial_capital
        self.trades = []
        self.data['positions'] = 0
        
    def calculate_macd(self):
        self.data['fast_ema'] = self.data[self.signal_price].ewm(span = self.fast_ema, adjust = False).mean()
        self.data['slow_ema'] = self.data[self.signal_price].ewm(span = self.slow_ema, adjust = False).mean()
        self.data['macd_line'] = self.data['fast_ema'] - self.data['slow_ema']
        self.data['signal_line'] = self.data['macd_line'].ewm(span = self.signal_line, adjust = False).mean()


    def generate_signals(self):
        """
        Generates trading signals based on MACD crossover.
        """
        threshold = 0.1
        self.data['signal'] = 0
        self.data['signal'] = np.where((self.data['macd_line'] - self.data['signal_line']) > threshold, 1, 0) # Buy signal
        self.data['signal'] = np.where((self.data['macd_line'] - self.data['signal_line']) < -threshold, 0, self.data['signal']) # Sell signal
        # Generation the position by shifting the signals
        #self.data['positions'] = self.data['signal'].shift(1).fillna(0)
        self.data['positions'] = self.data['signal']
        return self.data

    def backtest_strategy(self):
        """ Backtests the trading strategy. """
        self.data = self.data.copy()
        self.data['price'] = self.data[self.real_price]
        self.data['positions_diff'] = self.data['positions'].diff().fillna(0)
        
        self.data['cash'] = self.initial_capital
        self.data['holdings'] = 0.0
        self.data['total'] = self.initial_capital

        cash = self.initial_capital
        holdings = 0.0
        position = False
        buy_price = 0.0
        win_count = 0
        total_trades = 0

        for idx, row in self.data.iterrows():
            position_change = row['positions_diff']
            price = row['price']

            if position_change == 1 and not position:  # Enter long position
                holdings = cash * (1 - self.buy_fee_percent) / price
                cash = 0
                position = True
                buy_price = price

            elif position_change == -1 and position: # Exit long position
                cash = holdings * price * (1 - self.sell_fee_percent)
                holdings = 0
                position = False
                trade_return = ((price - buy_price) / buy_price) * 100
                self.trades.append(trade_return)
                total_trades += 1
                if trade_return > 0:
                    win_count += 1

            total = cash + (holdings * price if position else 0)
            self.data.at[idx, 'cash'] = float(cash)
            self.data.at[idx, 'holdings'] = float(holdings * price if position else 0)
            self.data.at[idx, 'total'] = float(total)

        self.results = self.data[['cash', 'holdings', 'total']]
        self.win_rate = (win_count / total_trades * 100) if total_trades > 0 else 0
        return self.data, self.trades


    def get_performance_metrics(self):
        """
        Calculates and returns performance metrics.
        """
        if self.results is None:
            print("Please run backtest_strategy() before calculating performance metrics.")
            return None

        total_return = (self.results['total'].iloc[-1] - self.initial_capital) / self.initial_capital * 100
        returns = self.results['total'].pct_change().fillna(0)
        annualized_return = ((1 + returns.mean()) ** 365 - 1) * 100  # Assuming daily returns
        annualized_volatility = returns.std() * np.sqrt(365) * 100

        # Calculating sharpe ratio
        periods_per_year = 8760

        # USA risk-free rate assumptions:
        # Assume an annual USA risk-free rate of 4%
        annual_rf_rate = 0.04  
        # Convert to a daily risk-free rate (using 365 days)
        daily_rf_rate = annual_rf_rate / 365  
        # Convert daily rate to an hourly rate (24 hours per day)
        hourly_rf_rate = daily_rf_rate / 24  

        # Calculate the excess hourly returns (return minus hourly risk-free rate)
        excess_returns = returns - hourly_rf_rate

        # Compute the mean and standard deviation of the hourly excess returns
        mean_hourly_excess = excess_returns.mean()
        std_hourly = returns.std()  # Using raw returns' std; you could also use excess_returns.std() if preferred

        # Annualize the Sharpe Ratio (even with less than one year of data)
        sharpe_ratio = (mean_hourly_excess / std_hourly) * np.sqrt(periods_per_year) if std_hourly != 0 else np.nan


        # Sortino Ratio
        target_return = hourly_rf_rate  
        downside_returns = returns.copy()
        downside_returns[downside_returns > target_return] = 0
        std_downside = downside_returns.std()
        annualized_downside_deviation = std_downside * np.sqrt(periods_per_year)

        # Annualized excess return (arithmetic) for Sortino ratio
        annualized_excess_return = mean_hourly_excess * periods_per_year

        # Sortino Ratio: excess return divided by annualized downside deviation
        sortino_ratio = annualized_excess_return / annualized_downside_deviation if annualized_downside_deviation != 0 else np.nan

 

        # sharpe_ratio = ( (returns.mean()* - 0.02) / returns.std()*np.sqrt(periods_per_year))  if returns.std() != 0 else np.nan

        max_drawdown = ((self.results['total'].cummax() - self.results['total']) / self.results['total'].cummax()).max() * 100

        metrics = {
            'Total Return (%)': total_return,
            'Annualized Return (%)': annualized_return,
            'Annualized Volatility (%)': annualized_volatility,
            'Sharpe Ratio': sharpe_ratio,
            'Sortion Ratio': sortino_ratio,
            'Max Drawdown (%)': max_drawdown,
            'Win Rate (%)' : self.win_rate
        }
        return metrics, self.results

    def print_trades(self):
        """
        Prints individual trade returns.
        """
        if not self.trades:
            print("No trades have been executed.")
            return
        for idx, trade_return in enumerate(self.trades, 1):
            print(f"Trade {idx}: Return = {trade_return:.2f}%")
        total_return = sum(self.trades)
        print(f"Total Return from trades: {total_return:.2f}%")

## run backterster

In [6]:
df_all = pd.concat((df_train, df_test), axis =0)

In [7]:
df_all

,open,high,low,close,volume,smoothed_data
2023-01-01 00:00:00,1198,1198,1198,1198,0.000000e+00,1197.638903
2023-01-01 04:00:00,1198,1198,1198,1198,0.000000e+00,1196.133578
2023-01-01 08:00:00,1198,1198,1198,1198,0.000000e+00,1193.987512
2023-01-01 12:00:00,1198,1198,1189,1192,4.366800e+04,1194.038570
2023-01-01 16:00:00,1192,1200,1191,1197,3.550640e+05,1196.564344
...,...,...,...,...,...,...
2024-11-05 04:00:00,2419,2444,2416,2441,1.097376e+07,2421.585036
2024-11-05 08:00:00,2442,2446,2429,2439,1.044846e+07,2430.875388
2024-11-05 12:00:00,2439,2479,2426,2439,1.650811e+07,2433.903299
2024-11-05 16:00:00,2439,2473,2400,2406,9.056088e+06,2428.736525


In [8]:
backtester = MACDBacktester(df_all, 12, 26, 9,signal_price=signal_price, real_price=real_price)
backtester.calculate_macd()
backtester.generate_signals()
backtester.backtest_strategy()
metrics, results = backtester.get_performance_metrics()
print(metrics)
backtester.print_trades()

C:\Users\arvin\AppData\Local\Temp\ipykernel_4216\3177299273.py:90: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '99.885' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  self.data.at[idx, 'total'] = float(total)
C:\Users\arvin\AppData\Local\Temp\ipykernel_4216\3177299273.py:88: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '103.93068822372811' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  self.data.at[idx, 'cash'] = float(cash)


{'Total Return (%)': 536.4961285642275, 'Annualized Return (%)': 19.59646944800333, 'Annualized Volatility (%)': 15.632796034625528, 'Sharpe Ratio': 5.557148696144018, 'Sortion Ratio': 9.591315671669122, 'Max Drawdown (%)': 15.001814355281168, 'Win Rate (%)': 53.84615384615385}
Trade 1: Return = 4.17%
Trade 2: Return = -0.32%
Trade 3: Return = 3.57%
Trade 4: Return = 11.75%
Trade 5: Return = 1.50%
Trade 6: Return = -1.69%
Trade 7: Return = -2.45%
Trade 8: Return = 5.79%
Trade 9: Return = -2.24%
Trade 10: Return = -2.62%
Trade 11: Return = 12.83%
Trade 12: Return = 2.81%
Trade 13: Return = -2.17%
Trade 14: Return = 13.78%
Trade 15: Return = 3.44%
Trade 16: Return = -3.02%
Trade 17: Return = 0.11%
Trade 18: Return = 4.02%
Trade 19: Return = 3.09%
Trade 20: Return = 0.48%
Trade 21: Return = 9.03%
Trade 22: Return = 1.50%
Trade 23: Return = 0.74%
Trade 24: Return = 1.66%
Trade 25: Return = -2.30%
Trade 26: Return = -0.89%
Trade 27: Return = -1.70%
Trade 28: Return = -0.28%
Trade 29: Return